# Prompt-Level Compressibility

This notebook asks what can be predicted before the reasoning trace is observed.

Unit of analysis: one full trace.

Target: `trace_high_token_compression`, created in `01_data_loading_engineering.ipynb` from the training-split trace compression threshold.

Predictors: problem-derived features and metadata only.

Models: dummy baseline, random forest, and gradient boosting. Hyperparameters are selected on a grouped validation split taken only from the training split, then the selected model is refit on the full training split and evaluated once on the test split.

In [78]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    count_share_table,
    latest_feature_build_dir,
    normalize_difficulty,
)

from src.reasoning_compression.modeling import (
    GRADIENT_BOOSTING_PARAM_GRID,
    RANDOM_FOREST_PARAM_GRID,
    random_forest_feature_importance,
    select_model_params,
)

pd.set_option("display.max_colwidth", None)


In [79]:
FULL_BUILD_DIR = latest_feature_build_dir(Path("../data/full_feature_builds"))
TRACE_TARGET = "trace_high_token_compression"
SPLIT_COL = "model_split"
GROUP_COL = "trace_id"

df_traces = pd.read_parquet(
    FULL_BUILD_DIR / "traces_features_full_labeled.parquet"
)
df_traces["difficulty"] = df_traces["difficulty"].map(normalize_difficulty)

required_columns = {TRACE_TARGET, SPLIT_COL, GROUP_COL}
missing_columns = required_columns.difference(df_traces.columns)
if missing_columns:
    raise ValueError(
        "Trace table is missing required columns: "
        f"{sorted(missing_columns)}. Re-run 01_data_loading_engineering.ipynb "
        "with RUN_FULL_BUILD = False."
    )

df_traces.shape

(228557, 27)

In [80]:
split_target_distribution = (
    df_traces
    .groupby(SPLIT_COL)[TRACE_TARGET]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .assign(share_pct=lambda df: (df["share"] * 100).round(2))
    .drop(columns="share")
)

split_target_distribution

,model_split,trace_high_token_compression,share_pct
0,test,0,75.12
1,test,1,24.88
2,train,0,75.00
3,train,1,25.00


**Prompt split check interpretation**

The target distribution is behaving as expected. The training split is exactly anchored at the 25% high-compression threshold, and the held-out test split is very close to the same class balance. This supports using balanced accuracy and ROC AUC as the main evaluation metrics rather than raw accuracy alone.

In [81]:
count_share_table(df_traces, "domain")

,n_rows,share_pct
domain,,
math,123333,53.96
science,61485,26.90
code,43739,19.14


## Feature Set

Prompt-level models use only information available before the model begins producing the reasoning trace: problem features and metadata.

In [82]:
numeric_features = [
    "problem_chars",
    "problem_tokens",
    "problem_math_symbol_share",
    "problem_question_mark_count",
]

binary_features = [
    "problem_has_multiple_choice",
    "problem_has_code_fence",
]

categorical_features = [
    "domain",
    "source",
    "difficulty",
]

feature_columns = numeric_features + binary_features + categorical_features

In [83]:
df_model = df_traces[
    feature_columns + [TRACE_TARGET, GROUP_COL, SPLIT_COL]
].dropna().copy()

X = df_model[feature_columns]
y = df_model[TRACE_TARGET]
groups = df_model[GROUP_COL]
model_split = df_model[SPLIT_COL]

train_mask = model_split == "train"
test_mask = model_split == "test"

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]
groups_train = groups.loc[train_mask]
groups_test = groups.loc[test_mask]

X_train.shape, X_test.shape

((182845, 9), (45712, 9))

In [84]:
len(set(groups_train).intersection(set(groups_test)))

0

## Modeling Utilities

Parameter grids, grouped validation tuning, and feature-importance helpers are imported from [`src.reasoning_compression.modeling`](../src/reasoning_compression). Preprocessing, fitting, and evaluation are defined directly with scikit-learn in this notebook.

In [85]:
rf_param_grid = RANDOM_FOREST_PARAM_GRID
gb_param_grid = GRADIENT_BOOSTING_PARAM_GRID


In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("bin", "passthrough", binary_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

## Baseline

In [87]:
prompt_dummy_model = DummyClassifier(strategy="most_frequent")
prompt_dummy_model.fit(X_train, y_train)

prompt_dummy_pred = prompt_dummy_model.predict(X_test)
prompt_dummy_proba = prompt_dummy_model.predict_proba(X_test)[:, 1]
prompt_dummy_metrics = {
    "accuracy": accuracy_score(y_test, prompt_dummy_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, prompt_dummy_pred),
    "roc_auc": roc_auc_score(y_test, prompt_dummy_proba),
}
pd.DataFrame([prompt_dummy_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.7512,0.5000,0.5000


## Random Forest

In [88]:
prompt_rf_selection_results = select_model_params(
    model_name="Random forest",
    model_class=RandomForestClassifier,
    param_grid=rf_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

display(
    prompt_rf_selection_results.style
    .format({metric: "{:.4f}" for metric in ["accuracy", "balanced_accuracy", "roc_auc"]})
    .set_properties(
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)


,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}",30000,24000,6000,0.6172,0.5749,0.6199
1,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 5, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}",30000,24000,6000,0.5838,0.5772,0.6192
2,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_estimators': 150, 'n_jobs': -1, 'random_state': 42}",30000,24000,6000,0.6185,0.5775,0.6187
3,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf': 5, 'n_estimators': 150, 'n_jobs': -1, 'random_state': 42}",30000,24000,6000,0.5832,0.5726,0.6175


In [89]:
prompt_rf_params = prompt_rf_selection_results.loc[0, "params"]
prompt_rf_model = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestClassifier(**prompt_rf_params)),
    ]
)
prompt_rf_model.fit(X_train, y_train)

prompt_rf_pred = prompt_rf_model.predict(X_test)
prompt_rf_proba = prompt_rf_model.predict_proba(X_test)[:, 1]
prompt_rf_metrics = {
    "accuracy": accuracy_score(y_test, prompt_rf_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, prompt_rf_pred),
    "roc_auc": roc_auc_score(y_test, prompt_rf_proba),
}
pd.DataFrame([prompt_rf_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.6294,0.6247,0.6807


In [90]:
random_forest_feature_importance(prompt_rf_model).head(20).round(4)

,feature,importance
0,num__problem_chars,0.3263
1,num__problem_math_symbol_share,0.2982
2,num__problem_tokens,0.2486
3,num__problem_question_mark_count,0.0312
4,cat__source_ai2-adapt-dev/openmath-2-math,0.0198
5,cat__domain_science,0.0197
6,cat__domain_math,0.0196
7,cat__source_stackexchange-physics,0.0118
8,cat__source_organic-chemistry-questions,0.0057
9,bin__problem_has_multiple_choice,0.0041


**Prompt feature importance interpretation**

The random forest relies mostly on prompt length and prompt composition: character count, token count, and math-symbol share dominate. Metadata and difficulty contribute much less. This is consistent with the prompt-level task: before any reasoning block is observed, the available signal is mostly about the type and surface structure of the problem.

## Gradient Boosting

In [91]:
prompt_gb_selection_results = select_model_params(
    model_name="Gradient boosting",
    model_class=HistGradientBoostingClassifier,
    param_grid=gb_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

display(
    prompt_gb_selection_results.style
    .format({metric: "{:.4f}" for metric in ["accuracy", "balanced_accuracy", "roc_auc"]})
    .set_properties(
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)


,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",30000,24000,6000,0.5270,0.5860,0.6144
1,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",30000,24000,6000,0.7618,0.5004,0.6115
2,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.1, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",30000,24000,6000,0.5307,0.5855,0.6110
3,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",30000,24000,6000,0.7623,0.5005,0.6108
4,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'random_state': 42}",30000,24000,6000,0.5037,0.5816,0.6103
5,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.1, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",30000,24000,6000,0.5043,0.5784,0.6093
6,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",30000,24000,6000,0.5103,0.5816,0.6093
7,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",30000,24000,6000,0.5155,0.5821,0.6092
8,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",30000,24000,6000,0.5118,0.5794,0.6092
9,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 150, 'max_leaf_nodes': 15, 'random_state': 42}",30000,24000,6000,0.7623,0.5000,0.6091


In [92]:
prompt_gb_params = prompt_gb_selection_results.loc[0, "params"]
prompt_gb_model = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", HistGradientBoostingClassifier(**prompt_gb_params)),
    ]
)
prompt_gb_model.fit(X_train, y_train)

prompt_gb_pred = prompt_gb_model.predict(X_test)
prompt_gb_proba = prompt_gb_model.predict_proba(X_test)[:, 1]
prompt_gb_metrics = {
    "accuracy": accuracy_score(y_test, prompt_gb_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, prompt_gb_pred),
    "roc_auc": roc_auc_score(y_test, prompt_gb_proba),
}
pd.DataFrame([prompt_gb_metrics]).style.format("{:.4f}")


,accuracy,balanced_accuracy,roc_auc
0,0.5312,0.6069,0.6444


## Model Comparison

In [ ]:
prompt_model_results = pd.DataFrame([
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Dummy",
        "selected_params": None,
        **prompt_dummy_metrics,
    },
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Random forest",
        "selected_params": prompt_rf_params,
        **prompt_rf_metrics,
    },
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Gradient boosting",
        "selected_params": prompt_gb_params,
        **prompt_gb_metrics,
    },
])

display(
    prompt_model_results.style
    .format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "roc_auc": "{:.4f}",
        "selected_params": lambda value: "None" if value is None else repr(value),
    })
    .set_properties(
        subset=["selected_params"],
        **{
            "white-space": "pre-wrap",
            "word-break": "break-word",
        }
    )
)


**Prompt model comparison interpretation**

The prompt-only models improve clearly over the dummy baseline in balanced accuracy and ROC AUC, but the effect is modest. Random forest is the strongest model in this run, with ROC AUC around 68%, while gradient boosting is weaker but has higher recall for the high-compression class. This is the expected pattern for the prompt-level question: the problem statement contains some predictive signal, but not enough to strongly determine compressibility before the reasoning trace is generated.

In [94]:
prompt_report_frames = []
for model_name, prediction in [
    ("Random forest", prompt_rf_pred),
    ("Gradient boosting", prompt_gb_pred),
]:
    report = classification_report(
        y_test,
        prediction,
        output_dict=True,
        zero_division=0,
    )
    report_frame = pd.DataFrame(report).T.reset_index(names="label")
    report_frame.insert(0, "model", model_name)
    prompt_report_frames.append(report_frame)

prompt_reports = pd.concat(prompt_report_frames, ignore_index=True)
prompt_reports.round(4)


,model,label,precision,recall,f1-score,support
0,Random forest,0,0.8327,0.6340,0.7199,34338.0000
1,Random forest,1,0.3577,0.6154,0.4524,11374.0000
2,Random forest,accuracy,0.6294,0.6294,0.6294,0.6294
3,Random forest,macro avg,0.5952,0.6247,0.5861,45712.0000
4,Random forest,weighted avg,0.7145,0.6294,0.6533,45712.0000
5,Gradient boosting,0,0.8503,0.4563,0.5939,34338.0000
6,Gradient boosting,1,0.3158,0.7575,0.4457,11374.0000
7,Gradient boosting,accuracy,0.5312,0.5312,0.5312,0.5312
8,Gradient boosting,macro avg,0.5830,0.6069,0.5198,45712.0000
9,Gradient boosting,weighted avg,0.7173,0.5312,0.5570,45712.0000
